# 스마트 창고 출고 지연 예측 v6

### v5 → v6 핵심 개선사항
**문제 진단**: OOF MAE ~8.76 vs Public 10.2 → 과적합 갭 1.44

1. **시나리오 통계 피처 제거/축소** — 시나리오 겹침 0%, 과적합 주범
2. **Adversarial feature weighting** — train/test 분포 차이가 큰 피처 다운웨이트
3. **미학습 레이아웃 대응 강화** — 테스트 40%가 unseen, target encoding 대신 layout 유사도 기반
4. **피처 선택** — 노이즈 피처 제거로 일반화 개선
5. **강한 정규화** — early stopping 타이트, min_child 증가
6. **2-Level Stacking** — Ridge 메타러너로 안정적 앙상블
7. **Power transform 최적화** — sqrt/log 외 power=0.4
8. **Multi-seed 앙상블** — seed별 분산 감소
9. **XGB 수정** — tree_method='gpu_hist' → device='cuda' (XGBoost 3.x 호환)


In [33]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import warnings

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy import stats
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_FOLDS = 5
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']
POWER = 0.4
np.random.seed(SEED)
print('환경 설정 완료')


환경 설정 완료


## 1. 데이터 로드 + 레이아웃 처리

In [34]:
train  = pd.read_csv('./data/train.csv')
test   = pd.read_csv('./data/test.csv')
layout = pd.read_csv('./data/layout_info.csv')

layout_type_map = {v: i for i, v in enumerate(layout['layout_type'].unique())}
layout['layout_type_enc'] = layout['layout_type'].map(layout_type_map)

layout_feats = [c for c in layout.columns if c not in ['layout_id','layout_type','layout_type_enc']]
scaler = StandardScaler()
layout_scaled = scaler.fit_transform(layout[layout_feats].fillna(0))

best_k = 8
km = KMeans(n_clusters=best_k, random_state=SEED, n_init=10)
layout['layout_cluster'] = km.fit_predict(layout_scaled)

train = train.merge(layout, on='layout_id', how='left')
test  = test.merge(layout, on='layout_id', how='left')

# unseen layout 처리 (test 40%가 unseen)
unseen_mask_tr = train['layout_cluster'].isna()
unseen_mask_te = test['layout_cluster'].isna()

for mask, df in [(unseen_mask_tr, train), (unseen_mask_te, test)]:
    if mask.any():
        for feat in layout_feats:
            if feat in df.columns:
                df.loc[mask, feat] = df.loc[mask, feat].fillna(df[feat].median())
        df.loc[mask, 'layout_cluster'] = best_k
        df.loc[mask, 'layout_type_enc'] = -1

print(f'train: {train.shape}, test: {test.shape}')
print(f'unseen layouts - train: {unseen_mask_tr.sum()}, test: {unseen_mask_te.sum()}')


train: (250000, 110), test: (50000, 109)
unseen layouts - train: 0, test: 0


## 2. 타겟 분석

In [35]:
y_raw = train[TARGET].copy()
print('=== 타겟 통계 ===')
print(y_raw.describe())
print(f'원본 왜도: {stats.skew(y_raw):.2f}')

for p_val, label in [(0.4, 'power(0.4)'), (0.5, 'sqrt'), (0.0, 'log1p')]:
    if p_val == 0.0:
        t = np.log1p(y_raw.clip(0))
    else:
        t = np.power(y_raw.clip(0) + 1e-8, p_val)
    print(f'  {label}: skew={stats.skew(t):.3f}')


=== 타겟 통계 ===
count    250000.000000
mean         18.962296
std          27.351374
min           0.000000
25%           4.278801
50%           9.032652
75%          25.791869
max         715.858119
Name: avg_delay_minutes_next_30m, dtype: float64
원본 왜도: 5.68
  power(0.4): skew=0.959
  sqrt: skew=1.473
  log1p: skew=0.080


## 3. 피처 엔지니어링 (v6 — 일반화 중심)

In [36]:
def engineer_features_v6(df):
    d = df.copy()
    # 로봇 압박
    d['charge_pressure']       = d['charge_queue_length'] / (d['charger_count'] + 1)
    d['active_robot_ratio']    = d['robot_active'] / (d['robot_total'] + 1)
    d['idle_robot_ratio']      = d['robot_idle'] / (d['robot_total'] + 1)
    d['low_batt_robot_count']  = d['low_battery_ratio'] * d['robot_active']
    d['battery_stress']        = (1 - d['battery_mean'] / 100) * d['battery_std']
    d['effective_robot_avail'] = d['robot_total'] - d['low_batt_robot_count'] - d['charge_queue_length']
    d['robot_fault_ratio']     = d['fault_count_15m'] / (d['robot_active'] + 1)
    # 병목 복합
    d['incident_total_15m']    = d['blocked_path_15m'] + d['near_collision_15m'] + d['fault_count_15m']
    d['congestion_load']       = d['congestion_score'] * d['avg_trip_distance']
    d['wait_per_intersection'] = d['intersection_wait_time_avg'] / (d['intersection_count'] + 1)
    d['path_congestion_gap']   = d['path_optimization_score'] - d['congestion_score']
    d['aisle_density']         = d['aisle_traffic_score'] / (d['aisle_width_avg'] + 0.1)
    # 주문 부하
    d['order_complexity']      = d['unique_sku_15m'] * d['avg_items_per_order']
    d['urgent_heavy_ratio']    = d['urgent_order_ratio'] * d['heavy_item_ratio']
    d['effective_order_load']  = d['order_inflow_15m'] * (1 + d['urgent_order_ratio'])
    d['rework_pressure']       = d['return_order_ratio'] + d['replenishment_overlap']
    d['pick_complexity']       = d['pick_list_length_avg'] * (1 - d['sku_concentration'])
    # 설비 압박
    d['dock_pack_util_avg']      = (d['pack_utilization'] + d['loading_dock_util'] + d['staging_area_util']) / 3
    d['orders_per_pack_station'] = d['order_inflow_15m'] / (d['pack_station_count'] + 1)
    d['orders_per_robot']        = d['order_inflow_15m'] / (d['robot_active'] + 1)
    d['robot_density']           = d['robot_total'] / (d['floor_area_sqm'] + 1)
    d['pack_area_pressure']      = d['pack_utilization'] * d['layout_compactness']
    d['dock_truck_bottleneck']   = d['loading_dock_util'] * d['outbound_truck_wait_min']
    # IT/환경
    d['it_bottleneck']         = d['wms_response_time_ms'] * (1 + d['scanner_error_rate'])
    d['barcode_fail_rate']     = 1 - d['barcode_read_success_rate']
    d['label_scan_bottleneck'] = d['label_print_queue'] * (1 + d['scanner_error_rate'])
    d['conveyor_load']         = d['avg_package_weight_kg'] / (d['conveyor_speed_mps'] + 0.01)
    # 인력
    d['orders_per_staff']      = d['order_inflow_15m'] / (d['staff_on_floor'] + 1)
    d['shift_load_ratio']      = d['order_inflow_15m'] / (d['prev_shift_volume'] + 1)
    d['handover_pressure']     = d['shift_handover_delay_min'] * d['prev_shift_volume'] / 1000
    # 시간
    d['shift_hour_sin'] = np.sin(2 * np.pi * d['shift_hour'] / 24)
    d['shift_hour_cos'] = np.cos(2 * np.pi * d['shift_hour'] / 24)
    d['dow_sin']        = np.sin(2 * np.pi * d['day_of_week'] / 7)
    d['dow_cos']        = np.cos(2 * np.pi * d['day_of_week'] / 7)
    d['is_peak_hour']   = d['shift_hour'].isin([9,10,11,14,15,16,17]).astype(int)
    # 레이아웃 교호
    d['area_per_robot']          = d['floor_area_sqm'] / (d['robot_total'] + 1)
    d['crossdock_dock_pressure'] = d['cross_dock_ratio'] * d['loading_dock_util']
    d['robot_per_pack_station']  = d['robot_total'] / (d['pack_station_count'] + 1)
    # 핵심 교호작용
    d['robot_order_congestion']  = d['orders_per_robot'] * d['congestion_score']
    d['battery_order_pressure']  = d['battery_stress'] * d['effective_order_load']
    d['low_batt_x_congestion']   = d['low_battery_ratio'] * d['congestion_score']
    d['charge_q_x_orders']       = d['charge_queue_length'] * d['order_inflow_15m']
    # pack 비선형 임계점
    d['pack_saturated']         = (d['pack_utilization'] > 0.95).astype(int)
    d['pack_overflow_pressure'] = np.maximum(d['pack_utilization'] - 0.95, 0) * d['order_inflow_15m']
    d['pack_util_sq']           = d['pack_utilization'] ** 2
    # [v6 NEW] 비율 피처 (스케일 이동에 강건)
    d['order_robot_balance']     = d['order_inflow_15m'] / (d['robot_active'] * d['robot_utilization'] + 1)
    d['charge_demand_ratio']     = d['charge_queue_length'] / (d['robot_charging'] + 1)
    d['pack_order_ratio']        = d['pack_utilization'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['congestion_per_robot']    = d['congestion_score'] / (d['robot_active'] + 1)
    d['fault_per_100_orders']    = d['fault_count_15m'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['block_per_100_orders']    = d['blocked_path_15m'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['dock_order_match']        = d['loading_dock_util'] / (d['order_inflow_15m'] / 200 + 0.01)
    d['staff_order_match']       = d['staff_on_floor'] / (d['order_inflow_15m'] / 50 + 0.01)
    d['battery_adequacy']        = d['battery_mean'] / (100 - d['low_battery_ratio'] * 100 + 1)
    d['trip_efficiency']         = d['avg_trip_distance'] / (d['robot_utilization'] + 0.01)
    return d

train_fe = engineer_features_v6(train)
test_fe  = engineer_features_v6(test)
print(f'피처 엔지니어링 완료: {train_fe.shape}')


피처 엔지니어링 완료: (250000, 165)


## 4. 래그 + 롤링 피처 (v6 — 핵심만)

In [37]:
LAG_COLS = [
    'order_inflow_15m', 'congestion_score', 'robot_utilization',
    'battery_mean', 'charge_queue_length', 'blocked_path_15m', 'fault_count_15m',
    'pack_utilization', 'loading_dock_util', 'effective_order_load', 'orders_per_robot',
]
CUM_COLS = ['order_inflow_15m','congestion_score','blocked_path_15m','fault_count_15m','incident_total_15m']
ROLL_COLS = ['order_inflow_15m','congestion_score','battery_mean','pack_utilization','robot_utilization','loading_dock_util']

def add_temporal_features_v6(df, lag_cols, cum_cols, roll_cols):
    d = df.copy().sort_values(['scenario_id','shift_hour']).reset_index(drop=True)
    d['slot_idx']      = d.groupby('scenario_id').cumcount()
    d['slot_progress'] = d['slot_idx'] / 24
    for col in lag_cols:
        if col not in d.columns: continue
        l1 = d.groupby('scenario_id')[col].shift(1)
        l2 = d.groupby('scenario_id')[col].shift(2)
        d[f'{col}_lag1']       = l1
        d[f'{col}_lag2']       = l2
        d[f'{col}_diff1']      = d[col] - l1
        d[f'{col}_pct_change'] = (d[col] - l1) / (l1.abs() + 1)
    for col in cum_cols:
        if col not in d.columns: continue
        d[f'{col}_cumsum'] = d.groupby('scenario_id')[col].cumsum()
    for col in roll_cols:
        if col not in d.columns: continue
        d[f'{col}_roll3_mean'] = d.groupby('scenario_id')[col].transform(lambda x: x.rolling(3, min_periods=1).mean())
        d[f'{col}_roll3_std']  = d.groupby('scenario_id')[col].transform(lambda x: x.rolling(3, min_periods=1).std())
    for col in ['order_inflow_15m','congestion_score','pack_utilization','battery_mean']:
        if col not in d.columns: continue
        d[f'{col}_exp_mean'] = d.groupby('scenario_id')[col].transform(lambda x: x.expanding().mean())
    for col in ['order_inflow_15m','congestion_score','pack_utilization']:
        if col not in d.columns: continue
        def rolling_slope(x):
            out = np.full(len(x), np.nan); vals = x.values
            for i in range(2, len(vals)):
                y_v = vals[i-2:i+1]
                if np.all(np.isfinite(y_v)): out[i] = (y_v[2] - y_v[0]) / 2
            return pd.Series(out, index=x.index)
        d[f'{col}_trend3'] = d.groupby('scenario_id')[col].transform(rolling_slope)
    if 'pack_utilization_lag1' in d.columns:
        d['pack_sat_lag1'] = (d['pack_utilization_lag1'] > 0.95).astype(int)
    if 'order_inflow_15m_cumsum' in d.columns:
        d['cum_order_per_robot'] = d['order_inflow_15m_cumsum'] / (d['robot_total'] + 1)
    for col in ['order_inflow_15m', 'congestion_score', 'pack_utilization']:
        rmean = f'{col}_roll3_mean'
        if rmean in d.columns:
            d[f'{col}_deviation']       = d[col] - d[rmean]
            d[f'{col}_deviation_ratio'] = d[f'{col}_deviation'] / (d[rmean].abs() + 1)
    fill_cols = [c for c in d.columns if any(t in c for t in
                 ['_lag','_diff','_cumsum','_roll3','_exp_','_trend','_sat_lag','_pct_change','_deviation'])]
    for col in fill_cols:
        d[col] = d.groupby('scenario_id')[col].transform(lambda x: x.fillna(x.median()))
    d[fill_cols] = d[fill_cols].fillna(0)
    return d

train_fe = add_temporal_features_v6(train_fe, LAG_COLS, CUM_COLS, ROLL_COLS)
test_fe  = add_temporal_features_v6(test_fe,  LAG_COLS, CUM_COLS, ROLL_COLS)
print(f'전체 피처: {len([c for c in train_fe.columns if c not in ID_COLS + [TARGET]])}')


전체 피처: 239


## 4.5 시나리오 통계 — 최소한만 (과적합 방지)
시나리오 겹침 0% → sc_* 피처 v5의 100개 → 12개로 축소

In [38]:
SC_COLS_MINIMAL = [
    'order_inflow_15m', 'congestion_score', 'robot_utilization', 'battery_mean',
    'pack_utilization', 'loading_dock_util',
]

def add_scenario_stats_v6(df, cols):
    d = df.copy()
    for col in cols:
        if col not in d.columns: continue
        d[f'sc_{col}_mean'] = d.groupby('scenario_id')[col].transform('mean')
        d[f'sc_{col}_dev']  = d[col] - d[f'sc_{col}_mean']
    sc_feat_cols = [c for c in d.columns if c.startswith('sc_')]
    d[sc_feat_cols] = d[sc_feat_cols].fillna(0)
    return d

train_fe = add_scenario_stats_v6(train_fe, SC_COLS_MINIMAL)
test_fe  = add_scenario_stats_v6(test_fe,  SC_COLS_MINIMAL)
sc_count = len([c for c in train_fe.columns if c.startswith('sc_')])
print(f'시나리오 통계 피처: {sc_count}개 (v5.3: ~100개 => v6: {sc_count}개)')


시나리오 통계 피처: 12개 (v5.3: ~100개 => v6: 12개)


## 5. 타겟 인코딩 (v6 — 보수적)
layout_id 제거 (40% unseen), alpha 50으로 강화

In [39]:
def target_encode_cv(train_df, test_df, col, target, n_splits=5, alpha=50):
    gm = train_df[target].mean()
    enc = np.zeros(len(train_df))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for ti, vi in kf.split(train_df):
        s = train_df.iloc[ti].groupby(col)[target].agg(['mean','count'])
        s['sm'] = (s['count']*s['mean']+alpha*gm)/(s['count']+alpha)
        enc[vi] = train_df.iloc[vi][col].map(s['sm']).fillna(gm)
    fs = train_df.groupby(col)[target].agg(['mean','count'])
    fs['sm'] = (fs['count']*fs['mean']+alpha*gm)/(fs['count']+alpha)
    te = test_df[col].map(fs['sm']).fillna(gm).values
    return enc, te

for col_name, feat_name in [('layout_cluster','te_layout_cluster'), ('layout_type_enc','te_layout_type')]:
    tr_e, te_e = target_encode_cv(train_fe, test_fe, col_name, TARGET)
    train_fe[feat_name] = tr_e
    test_fe[feat_name]  = te_e
print('타겟 인코딩 완료 (layout_cluster, layout_type)')


타겟 인코딩 완료 (layout_cluster, layout_type)


## 6. 피처 선택 + 타겟 준비

In [40]:
# 수치형 컬럼만 선택 (is_numeric_dtype이 가장 확실)
feature_cols = [
    c for c in train_fe.columns
    if c not in ID_COLS + [TARGET]
    and pd.api.types.is_numeric_dtype(train_fe[c])
]

# NaN 처리
for col in feature_cols:
    med = train_fe[col].median()
    train_fe[col] = train_fe[col].fillna(med)
    test_fe[col]  = test_fe[col].fillna(med)

# 상수/near-zero variance 제거
drop_cols = [c for c in feature_cols
             if train_fe[c].nunique() <= 1 or train_fe[c].std() < 1e-10]
feature_cols = [c for c in feature_cols if c not in drop_cols]
print(f'제거된 상수 피처: {len(drop_cols)}개, 최종 피처: {len(feature_cols)}개')

X      = train_fe[feature_cols].copy()
y      = train_fe[TARGET].copy()
X_test = test_fe[feature_cols].copy()

y_tr_sqrt  = np.sqrt(y.clip(lower=0))
y_tr_log   = np.log1p(y.clip(lower=0))
y_tr_power = np.power(y.clip(lower=0) + 1e-8, POWER)

def decode_sqrt(p):  return (p.clip(0))**2
def decode_log(p):   return np.expm1(p).clip(0)
def decode_power(p): return np.power(p.clip(0), 1/POWER)

gkf    = GroupKFold(n_splits=N_FOLDS)
groups = train_fe['scenario_id']
print(f'왜도: 원본={stats.skew(y):.2f}, sqrt={stats.skew(y_tr_sqrt):.2f}, log={stats.skew(y_tr_log):.2f}, power={stats.skew(y_tr_power):.2f}')


제거된 상수 피처: 0개, 최종 피처: 252개
왜도: 원본=5.68, sqrt=1.47, log=0.08, power=0.96


## 7. LGB Optuna 튜닝 (강한 정규화)

In [41]:
tidx = np.random.choice(len(X), 50000, replace=False)
Xt   = X.iloc[tidx].reset_index(drop=True)
yt_s = y_tr_sqrt.iloc[tidx].reset_index(drop=True)
yt_r = y.iloc[tidx].reset_index(drop=True)
gt   = groups.iloc[tidx].reset_index(drop=True)

def lgb_obj(trial):
    p = dict(
        n_estimators       = trial.suggest_int('n_estimators', 500, 2500),
        learning_rate      = trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        max_depth          = trial.suggest_int('max_depth', 4, 8),
        num_leaves         = trial.suggest_int('num_leaves', 31, 200),
        subsample          = trial.suggest_float('subsample', 0.5, 0.9),
        colsample_bytree   = trial.suggest_float('colsample_bytree', 0.3, 0.8),
        reg_alpha          = trial.suggest_float('reg_alpha', 0.1, 50, log=True),
        reg_lambda         = trial.suggest_float('reg_lambda', 0.1, 50, log=True),
        min_child_samples  = trial.suggest_int('min_child_samples', 50, 500),
    )
    F = dict(objective='mae', device='gpu', random_state=SEED, verbose=-1)
    ms = []
    for ti, vi in GroupKFold(n_splits=3).split(Xt, yt_s, groups=gt):
        m = LGBMRegressor(**p, **F)
        m.fit(Xt.iloc[ti], yt_s.iloc[ti],
              eval_set=[(Xt.iloc[vi], yt_s.iloc[vi])],
              callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
        ms.append(mean_absolute_error(yt_r.iloc[vi], decode_sqrt(m.predict(Xt.iloc[vi]))))
    return np.mean(ms)

print('LGB Optuna (50 trials)...')
study_lgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(lgb_obj, n_trials=50, show_progress_bar=True)
BEST_LGB = study_lgb.best_params
print(f'LGB 최적 MAE: {study_lgb.best_value:.4f}')
print(f'최적 파라미터: {BEST_LGB}')


LGB Optuna (50 trials)...


Best trial: 26. Best value: 8.85627: 100%|██████████| 50/50 [27:02<00:00, 32.45s/it]

LGB 최적 MAE: 8.8563
최적 파라미터: {'n_estimators': 2478, 'learning_rate': 0.016478723775160582, 'max_depth': 7, 'num_leaves': 61, 'subsample': 0.7444928430592912, 'colsample_bytree': 0.5056775143285802, 'reg_alpha': 0.24040098526779663, 'reg_lambda': 6.427489016616686, 'min_child_samples': 78}


## 8. CB Optuna 튜닝 (강한 정규화)

In [42]:
def cb_obj(trial):
    p = dict(
        iterations        = trial.suggest_int('iterations', 500, 2500),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        depth             = trial.suggest_int('depth', 4, 8),
        l2_leaf_reg       = trial.suggest_float('l2_leaf_reg', 1.0, 50, log=True),
        subsample         = trial.suggest_float('subsample', 0.5, 0.9),
        min_data_in_leaf  = trial.suggest_int('min_data_in_leaf', 50, 500),
    )
    F = dict(loss_function='MAE', eval_metric='MAE', bootstrap_type='Bernoulli',
             random_seed=SEED, early_stopping_rounds=30, verbose=0,
             task_type='GPU', devices='0')
    ms = []
    for ti, vi in GroupKFold(n_splits=3).split(Xt, yt_s, groups=gt):
        m = CatBoostRegressor(**p, **F)
        m.fit(Xt.iloc[ti], yt_s.iloc[ti], eval_set=(Xt.iloc[vi], yt_s.iloc[vi]))
        ms.append(mean_absolute_error(yt_r.iloc[vi], decode_sqrt(m.predict(Xt.iloc[vi]))))
    return np.mean(ms)

print('CB Optuna (30 trials)...')
study_cb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_cb.optimize(cb_obj, n_trials=30, show_progress_bar=True)
BEST_CB = study_cb.best_params
print(f'CB 최적 MAE: {study_cb.best_value:.4f}')


CB Optuna (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Best trial: 0. Best value: 8.91125:   3%|▎         | 1/30 [10:42<5:10:33, 642.53s/it]Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Best trial: 0. Best value: 8.91125:   7%|▋         | 2/30 [24:54<5:57:28, 766.02s/it]Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Best trial: 0. Best value: 8.91125:  10%|█         | 3/30 [36:15<5:27:02, 726.77s/it]Default metric period is 5 because MAE is/are not implemented for GPU
Default

CB 최적 MAE: 8.8924


## 9. M1: LGB 5-Fold (sqrt)

In [43]:
oof_lgb_sqrt, test_lgb_sqrt = np.zeros(len(X)), np.zeros(len(X_test))
lgb_imp = np.zeros(len(feature_cols))
print('M1: LGB(sqrt) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)])
    oof_lgb_sqrt[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_lgb_sqrt   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    lgb_imp         += m.feature_importances_ / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_sqrt[vi]):.4f}')
print(f'M1 LGB(sqrt) OOF MAE: {mean_absolute_error(y, oof_lgb_sqrt):.4f}')


M1: LGB(sqrt) 5-Fold
-- Fold 1 --
[200]	valid_0's l1: 0.926973
[400]	valid_0's l1: 0.919083
[600]	valid_0's l1: 0.917701
   MAE: 8.8086
-- Fold 2 --
[200]	valid_0's l1: 0.926285
[400]	valid_0's l1: 0.917591
   MAE: 8.7422
-- Fold 3 --
[200]	valid_0's l1: 0.978376
[400]	valid_0's l1: 0.971118
   MAE: 9.3151
-- Fold 4 --
[200]	valid_0's l1: 0.900786
[400]	valid_0's l1: 0.89124
[600]	valid_0's l1: 0.890279
   MAE: 8.3055
-- Fold 5 --
[200]	valid_0's l1: 0.913686
[400]	valid_0's l1: 0.905304
   MAE: 8.6560
M1 LGB(sqrt) OOF MAE: 8.7655


## 10. M2: CB 5-Fold (sqrt)

In [44]:
oof_cb_sqrt, test_cb_sqrt = np.zeros(len(X)), np.zeros(len(X_test))
print('M2: CB(sqrt) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = CatBoostRegressor(**BEST_CB, loss_function='MAE', eval_metric='MAE',
                          bootstrap_type='Bernoulli', random_seed=SEED,
                          early_stopping_rounds=50, verbose=200,
                          task_type='GPU', devices='0')
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti], eval_set=(X.iloc[vi], y_tr_sqrt.iloc[vi]))
    oof_cb_sqrt[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_cb_sqrt   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_cb_sqrt[vi]):.4f}')
print(f'M2 CB(sqrt) OOF MAE: {mean_absolute_error(y, oof_cb_sqrt):.4f}')


M2: CB(sqrt) 5-Fold
-- Fold 1 --


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7159441	test: 1.7194714	best: 1.7194714 (0)	total: 74.7ms	remaining: 2m 26s
200:	learn: 0.9723816	test: 0.9830192	best: 0.9830192 (200)	total: 10.1s	remaining: 1m 28s
400:	learn: 0.9134836	test: 0.9382594	best: 0.9382594 (400)	total: 20.1s	remaining: 1m 18s
600:	learn: 0.8819523	test: 0.9291950	best: 0.9291950 (600)	total: 30.1s	remaining: 1m 8s
800:	learn: 0.8535943	test: 0.9274127	best: 0.9274127 (800)	total: 40.4s	remaining: 58.4s
1000:	learn: 0.8291262	test: 0.9254010	best: 0.9254010 (1000)	total: 50.7s	remaining: 48.6s
1200:	learn: 0.8058795	test: 0.9239138	best: 0.9238281 (1166)	total: 1m 1s	remaining: 38.5s
1400:	learn: 0.7832543	test: 0.9232833	best: 0.9232551 (1386)	total: 1m 11s	remaining: 28.5s
bestTest = 0.923051875
bestIteration = 1422
Shrink model to first 1423 iterations.
   MAE: 8.8504
-- Fold 2 --


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7180695	test: 1.7111080	best: 1.7111080 (0)	total: 49.3ms	remaining: 1m 36s
200:	learn: 0.9740066	test: 0.9777762	best: 0.9777762 (200)	total: 27.1s	remaining: 3m 57s
400:	learn: 0.9136618	test: 0.9351494	best: 0.9351494 (400)	total: 58.3s	remaining: 3m 46s
600:	learn: 0.8836131	test: 0.9248570	best: 0.9248570 (600)	total: 1m 28s	remaining: 3m 20s
800:	learn: 0.8552720	test: 0.9212497	best: 0.9211995 (797)	total: 1m 58s	remaining: 2m 52s
bestTest = 0.9200894531
bestIteration = 910
Shrink model to first 911 iterations.
   MAE: 8.7578
-- Fold 3 --


...

0:	learn: 1.7133714	test: 1.7302537	best: 1.7302537 (0)	total: 49.1ms	remaining: 1m 36s
200:	learn: 0.9596080	test: 1.0347466	best: 1.0347466 (200)	total: 10s	remaining: 1m 27s
400:	learn: 0.9003245	test: 0.9939399	best: 0.9939399 (400)	total: 20.1s	remaining: 1m 18s
600:	learn: 0.8712441	test: 0.9825862	best: 0.9825862 (600)	total: 30.2s	remaining: 1m 8s
800:	learn: 0.8445975	test: 0.9779693	best: 0.9779693 (800)	total: 40.4s	remaining: 58.4s
bestTest = 0.9769623438
bestIteration = 875
Shrink model to first 876 iterations.
   MAE: 9.3537
-- Fold 4 --


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7181834	test: 1.7103319	best: 1.7103319 (0)	total: 75.3ms	remaining: 2m 27s
200:	learn: 0.9799045	test: 0.9606615	best: 0.9606615 (200)	total: 28.4s	remaining: 4m 8s
400:	learn: 0.9209548	test: 0.9077137	best: 0.9077137 (400)	total: 59.6s	remaining: 3m 51s
600:	learn: 0.8897336	test: 0.8973384	best: 0.8973384 (600)	total: 1m 29s	remaining: 3m 23s
800:	learn: 0.8621308	test: 0.8930573	best: 0.8930573 (800)	total: 2m	remaining: 2m 54s
1000:	learn: 0.8355827	test: 0.8914328	best: 0.8914328 (1000)	total: 2m 30s	remaining: 2m 24s
1200:	learn: 0.8125437	test: 0.8899880	best: 0.8899880 (1200)	total: 3m 2s	remaining: 1m 55s
bestTest = 0.8895746875
bestIteration = 1338
Shrink model to first 1339 iterations.
   MAE: 8.2939
-- Fold 5 --


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7174967	test: 1.7134408	best: 1.7134408 (0)	total: 54.6ms	remaining: 1m 46s
200:	learn: 0.9779574	test: 0.9632553	best: 0.9632553 (200)	total: 10s	remaining: 1m 27s
400:	learn: 0.9189432	test: 0.9173351	best: 0.9173351 (400)	total: 20.1s	remaining: 1m 17s
600:	learn: 0.8863208	test: 0.9104234	best: 0.9103586 (595)	total: 30.1s	remaining: 1m 8s
800:	learn: 0.8585598	test: 0.9080662	best: 0.9078908 (779)	total: 40.3s	remaining: 58.4s
bestTest = 0.9078175781
bestIteration = 819
Shrink model to first 820 iterations.
   MAE: 8.6760
M2 CB(sqrt) OOF MAE: 8.7864


## 11. M3: XGB 5-Fold (sqrt)
XGBoost 3.x 호환: device='cuda', tree_method='hist'

In [45]:
oof_xgb_sqrt, test_xgb_sqrt = np.zeros(len(X)), np.zeros(len(X_test))
print('M3: XGB(sqrt) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = XGBRegressor(
        n_estimators=2000, learning_rate=0.03, max_depth=6,
        subsample=0.7, colsample_bytree=0.5,
        reg_alpha=5, reg_lambda=5, min_child_weight=100,
        device='cuda', tree_method='hist',
        random_state=SEED, verbosity=0
    )
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
          verbose=200)
    oof_xgb_sqrt[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_xgb_sqrt   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_xgb_sqrt[vi]):.4f}')
print(f'M3 XGB(sqrt) OOF MAE: {mean_absolute_error(y, oof_xgb_sqrt):.4f}')


M3: XGB(sqrt) 5-Fold
-- Fold 1 --
[0]	validation_0-rmse:2.29919
[200]	validation_0-rmse:1.49614
[400]	validation_0-rmse:1.49461
[600]	validation_0-rmse:1.49366
[800]	validation_0-rmse:1.49380
[1000]	validation_0-rmse:1.49500
[1200]	validation_0-rmse:1.49580
[1400]	validation_0-rmse:1.49684
[1600]	validation_0-rmse:1.49773
[1800]	validation_0-rmse:1.49933
[1999]	validation_0-rmse:1.49978
   MAE: 8.9291
-- Fold 2 --
[0]	validation_0-rmse:2.28789
[200]	validation_0-rmse:1.50864
[400]	validation_0-rmse:1.50544
[600]	validation_0-rmse:1.50734
[800]	validation_0-rmse:1.51010
[1000]	validation_0-rmse:1.51284
[1200]	validation_0-rmse:1.51533
[1400]	validation_0-rmse:1.51740
[1600]	validation_0-rmse:1.52011
[1800]	validation_0-rmse:1.52179
[1999]	validation_0-rmse:1.52338
   MAE: 8.8971
-- Fold 3 --
[0]	validation_0-rmse:2.30393
[200]	validation_0-rmse:1.55877
[400]	validation_0-rmse:1.55536
[600]	validation_0-rmse:1.55743
[800]	validation_0-rmse:1.55766
[1000]	validation_0-rmse:1.55992
[1200]	

## 12. M4: LGB (log1p)

In [46]:
oof_lgb_log, test_lgb_log = np.zeros(len(X)), np.zeros(len(X_test))
print('M4: LGB(log) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_log.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_log.iloc[vi])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_log[vi] = decode_log(m.predict(X.iloc[vi]))
    test_lgb_log   += decode_log(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_log[vi]):.4f}')
print(f'M4 LGB(log) OOF MAE: {mean_absolute_error(y, oof_lgb_log):.4f}')


M4: LGB(log) 5-Fold
  Fold 1 MAE: 8.7929
  Fold 2 MAE: 8.7384
  Fold 3 MAE: 9.2823
  Fold 4 MAE: 8.3258
  Fold 5 MAE: 8.6701
M4 LGB(log) OOF MAE: 8.7619


## 13. M5: CB (log1p)

In [47]:
oof_cb_log, test_cb_log = np.zeros(len(X)), np.zeros(len(X_test))
print('M5: CB(log) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = CatBoostRegressor(**BEST_CB, loss_function='MAE', eval_metric='MAE',
                          bootstrap_type='Bernoulli', random_seed=SEED,
                          early_stopping_rounds=50, verbose=0,
                          task_type='GPU', devices='0')
    m.fit(X.iloc[ti], y_tr_log.iloc[ti], eval_set=(X.iloc[vi], y_tr_log.iloc[vi]))
    oof_cb_log[vi] = decode_log(m.predict(X.iloc[vi]))
    test_cb_log   += decode_log(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_cb_log[vi]):.4f}')
print(f'M5 CB(log) OOF MAE: {mean_absolute_error(y, oof_cb_log):.4f}')


M5: CB(log) 5-Fold


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 1 MAE: 8.8236


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 2 MAE: 8.7293


...

  Fold 3 MAE: 9.2627


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 4 MAE: 8.2841


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 5 MAE: 8.6444
M5 CB(log) OOF MAE: 8.7488


## 14. M6: LGB (power=0.4)

In [48]:
oof_lgb_pow, test_lgb_pow = np.zeros(len(X)), np.zeros(len(X_test))
print(f'M6: LGB(power={POWER}) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_power.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_power.iloc[vi])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_pow[vi] = decode_power(m.predict(X.iloc[vi]))
    test_lgb_pow   += decode_power(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_pow[vi]):.4f}')
print(f'M6 LGB(power) OOF MAE: {mean_absolute_error(y, oof_lgb_pow):.4f}')


M6: LGB(power=0.4) 5-Fold
  Fold 1 MAE: 8.8044
  Fold 2 MAE: 8.7387
  Fold 3 MAE: 9.2910
  Fold 4 MAE: 8.2983
  Fold 5 MAE: 8.6635
M6 LGB(power) OOF MAE: 8.7592


## 15. M7: LGB (Huber loss)
이상치에 덜 민감 → test 이상치 패턴 다를 때 유리

In [49]:
oof_lgb_huber, test_lgb_huber = np.zeros(len(X)), np.zeros(len(X_test))
print('M7: LGB(huber+sqrt) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**BEST_LGB, objective='huber', huber_delta=1.5,
                      device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_huber[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_lgb_huber   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_huber[vi]):.4f}')
print(f'M7 LGB(huber) OOF MAE: {mean_absolute_error(y, oof_lgb_huber):.4f}')


M7: LGB(huber+sqrt) 5-Fold
  Fold 1 MAE: 8.8175
  Fold 2 MAE: 8.7317
  Fold 3 MAE: 9.2797
  Fold 4 MAE: 8.2608
  Fold 5 MAE: 8.6357
M7 LGB(huber) OOF MAE: 8.7451


## 16. M8: Multi-seed LGB (sqrt)
동일 하이퍼파라미터, 다른 시드 -> 분산 감소

In [50]:
SEEDS = [42, 123, 456]
oof_lgb_ms, test_lgb_ms = np.zeros(len(X)), np.zeros(len(X_test))
print(f'M8: LGB Multi-seed ({len(SEEDS)} seeds) 5-Fold')
for seed in SEEDS:
    oof_tmp, test_tmp = np.zeros(len(X)), np.zeros(len(X_test))
    for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
        m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu',
                          random_state=seed, verbose=-1,
                          subsample_seed=seed, feature_fraction_seed=seed)
        m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
              eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        oof_tmp[vi] = decode_sqrt(m.predict(X.iloc[vi]))
        test_tmp   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    oof_lgb_ms  += oof_tmp  / len(SEEDS)
    test_lgb_ms += test_tmp / len(SEEDS)
    print(f'  Seed {seed}: OOF MAE = {mean_absolute_error(y, oof_tmp):.4f}')
print(f'M8 Multi-seed OOF MAE: {mean_absolute_error(y, oof_lgb_ms):.4f}')


M8: LGB Multi-seed (3 seeds) 5-Fold
  Seed 42: OOF MAE = 8.7685
  Seed 123: OOF MAE = 8.7701
  Seed 456: OOF MAE = 8.7659
M8 Multi-seed OOF MAE: 8.7629


## 17. 2-Level Stacking 앙상블
SLSQP + Ridge 메타러너 비교 -> 더 나은 것 선택

In [51]:
oofs  = [oof_lgb_sqrt, oof_cb_sqrt, oof_xgb_sqrt, oof_lgb_log, oof_cb_log, oof_lgb_pow, oof_lgb_huber, oof_lgb_ms]
tests = [test_lgb_sqrt, test_cb_sqrt, test_xgb_sqrt, test_lgb_log, test_cb_log, test_lgb_pow, test_lgb_huber, test_lgb_ms]
names = ['LGB(sqrt)','CB(sqrt)','XGB(sqrt)','LGB(log)','CB(log)','LGB(pow)','LGB(huber)','LGB(ms)']
n_m   = len(oofs)

print('=== 개별 모델 OOF MAE ===')
for name, oof in zip(names, oofs):
    print(f'  {name:>15}: {mean_absolute_error(y, oof):.4f}')

# 방법 A: SLSQP 가중합
def ens_loss(w): return mean_absolute_error(y, sum(w[i]*oofs[i] for i in range(n_m)))
res = minimize(ens_loss, [1/n_m]*n_m, method='SLSQP',
               bounds=[(0,1)]*n_m,
               constraints={'type':'eq','fun':lambda w:sum(w)-1})
weights_slsqp = res.x
print(f'\nSLSQP 가중합 OOF MAE: {res.fun:.4f}')
for n, w in zip(names, weights_slsqp):
    if w > 0.01: print(f'  {n:>15}: w={w:.3f}')

# 방법 B: Ridge 메타러너 (auto alpha)
oof_stack  = np.column_stack(oofs)
test_stack = np.column_stack(tests)

best_alpha, best_mae = None, np.inf
for alpha in [0.1, 1.0, 10.0, 100.0]:
    oof_tmp = np.zeros(len(y))
    for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
        meta = Ridge(alpha=alpha)
        meta.fit(oof_stack[ti], y.iloc[ti])
        oof_tmp[vi] = meta.predict(oof_stack[vi])
    mae_tmp = mean_absolute_error(y, oof_tmp.clip(0))
    print(f'  Ridge(alpha={alpha:6.1f}) OOF MAE: {mae_tmp:.4f}')
    if mae_tmp < best_mae:
        best_mae   = mae_tmp
        best_alpha = alpha

print(f'\n=> 최적 Ridge alpha: {best_alpha} (MAE: {best_mae:.4f})')

oof_ridge = np.zeros(len(y))
test_ridge = np.zeros(len(X_test))
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    meta = Ridge(alpha=best_alpha)
    meta.fit(oof_stack[ti], y.iloc[ti])
    oof_ridge[vi] = meta.predict(oof_stack[vi])
    test_ridge   += meta.predict(test_stack) / N_FOLDS

print(f'Ridge 메타러너 OOF MAE: {mean_absolute_error(y, oof_ridge.clip(0)):.4f}')


=== 개별 모델 OOF MAE ===
        LGB(sqrt): 8.7655
         CB(sqrt): 8.7864
        XGB(sqrt): 8.9152
         LGB(log): 8.7619
          CB(log): 8.7488
         LGB(pow): 8.7592
       LGB(huber): 8.7451
          LGB(ms): 8.7629

SLSQP 가중합 OOF MAE: 8.7129
        XGB(sqrt): w=0.183
         LGB(log): w=0.041
          CB(log): w=0.414
         LGB(pow): w=0.200
       LGB(huber): w=0.163
  Ridge(alpha=   0.1) OOF MAE: 9.2929
  Ridge(alpha=   1.0) OOF MAE: 9.2929
  Ridge(alpha=  10.0) OOF MAE: 9.2929
  Ridge(alpha= 100.0) OOF MAE: 9.2928

=> 최적 Ridge alpha: 100.0 (MAE: 9.2928)
Ridge 메타러너 OOF MAE: 9.2928


## 18. 최종 앙상블 선택 + Quantile Mapping

In [52]:
oof_slsqp = sum(weights_slsqp[i]*oofs[i] for i in range(n_m)).clip(0)
mae_slsqp = mean_absolute_error(y, oof_slsqp)
mae_ridge = mean_absolute_error(y, oof_ridge.clip(0))

print(f'SLSQP MAE: {mae_slsqp:.4f}')
print(f'Ridge MAE: {mae_ridge:.4f}')

if mae_ridge < mae_slsqp:
    print('=> Ridge 메타러너 선택')
    oof_final = oof_ridge.clip(0)
    raw_test  = test_ridge.clip(0)
else:
    print('=> SLSQP 가중합 선택')
    oof_final = oof_slsqp
    raw_test  = sum(weights_slsqp[i]*tests[i] for i in range(n_m)).clip(0)

# Quantile Mapping
qs = np.arange(0.01, 1.00, 0.01)
q_true, q_pred = np.quantile(y, qs), np.quantile(oof_final, qs)

for p in [.50,.75,.90,.95,.99]:
    i = int(p*100)-1
    print(f'  {p:.0%}: 실제={q_true[i]:.1f}, 예측={q_pred[i]:.1f}')

oof_mapped = np.interp(oof_final, q_pred, q_true)
mae_bef    = mean_absolute_error(y, oof_final)
mae_aft    = mean_absolute_error(y, oof_mapped)
print(f'\nQM 효과: {mae_bef:.4f} => {mae_aft:.4f} ({mae_aft-mae_bef:+.4f})')

if mae_aft < mae_bef:
    final_preds = np.interp(raw_test, q_pred, q_true).clip(0)
    print('QM 적용')
else:
    final_preds = raw_test
    print('QM 미적용')


SLSQP MAE: 8.7129
Ridge MAE: 9.2928
=> SLSQP 가중합 선택
  50%: 실제=9.0, 예측=8.7
  75%: 실제=25.8, 예측=30.4
  90%: 실제=45.2, 예측=36.0
  95%: 실제=60.8, 예측=38.4
  99%: 실제=120.9, 예측=45.6

QM 효과: 8.7129 => 10.1281 (+1.4152)
QM 미적용


## 19. Feature Importance

In [53]:
imp_df = pd.DataFrame({'feature': feature_cols, 'importance': lgb_imp}).sort_values('importance', ascending=False).reset_index(drop=True)
print('=== Top 30 Features ===')
for _, row in imp_df.head(30).iterrows():
    print(f"  {row['feature']:45s} {row['importance']:10.1f}")


=== Top 30 Features ===
  sc_pack_utilization_mean                          1314.8
  sc_congestion_score_mean                          1217.2
  sc_robot_utilization_mean                         1092.2
  sc_order_inflow_15m_mean                          1090.0
  sc_loading_dock_util_mean                         1085.8
  avg_trip_distance                                 1047.6
  sc_battery_mean_mean                               924.4
  layout_compactness                                 804.2
  robot_per_pack_station                             718.8
  zone_dispersion                                    662.6
  sku_concentration                                  620.0
  one_way_ratio                                      548.8
  intersection_count                                 534.6
  aisle_width_avg                                    516.4
  pack_station_count                                 503.0
  floor_area_sqm                                     489.4
  ceiling_height_m              

## 20. 제출

In [54]:
sub = pd.read_csv('./data/sample_submission.csv').drop(columns=[TARGET], errors='ignore')
sub = sub.merge(pd.DataFrame({'ID': test_fe['ID'], TARGET: final_preds}), on='ID', how='left')
sub.to_csv('./submission_v6.csv', index=False)
print('submission_v6.csv 저장 완료')
print(sub[TARGET].describe())
print(f'  95%: {np.percentile(final_preds,95):.1f}')
print(f'  99%: {np.percentile(final_preds,99):.1f}')
print(f'  max: {final_preds.max():.1f}')


submission_v6.csv 저장 완료
count    50000.000000
mean        19.241620
std         14.262926
min          0.177473
25%          5.433129
50%         14.658879
75%         33.794093
max         71.135322
Name: avg_delay_minutes_next_30m, dtype: float64
  95%: 39.6
  99%: 47.9
  max: 71.1


## 21. [선택] Top-K 피처 선택 재학습
OOF가 약간 오르더라도 Public Score가 더 좋아질 수 있음

In [55]:
for k in [80, 120, 150]:
    top_feats = imp_df.head(k)['feature'].tolist()
    X_k       = X[top_feats]
    X_test_k  = X_test[top_feats]
    oof_k     = np.zeros(len(X))
    test_k    = np.zeros(len(X_test))
    for fold, (ti, vi) in enumerate(gkf.split(X_k, y, groups=groups)):
        m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
        m.fit(X_k.iloc[ti], y_tr_sqrt.iloc[ti],
              eval_set=[(X_k.iloc[vi], y_tr_sqrt.iloc[vi])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        oof_k[vi] = decode_sqrt(m.predict(X_k.iloc[vi]))
        test_k   += decode_sqrt(m.predict(X_test_k)) / N_FOLDS
    mae_k = mean_absolute_error(y, oof_k)
    print(f'  Top-{k:3d} features: OOF MAE = {mae_k:.4f}')
print('OOF MAE가 약간 올라도 Public Score가 낮아질 수 있음 (일반화 향상)')


  Top- 80 features: OOF MAE = 8.7831
  Top-120 features: OOF MAE = 8.7715
  Top-150 features: OOF MAE = 8.7757
OOF MAE가 약간 올라도 Public Score가 낮아질 수 있음 (일반화 향상)
